### Management wants to know:
1.How many total patients do we have?
3.Patient distribution by gender.
4.Patient distribution by age group .
5.Which locations have the most patients?

In [3]:
from pyspark.sql.functions import col , when , count , sum , max ,min

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 5, Finished, Available, Finished, False)

In [4]:

patients = spark.table("Silver_LH.dbo.silver_patients")

admissions = spark.table("Silver_LH.dbo.silver_admissions_valid")

medicaltest = spark.table("Silver_LH.dbo.silver_medicaltest_valid")

display(patients)

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec59985b-5370-467b-ae99-0ad3f9277e48)

In [5]:
patients.printSchema()

admissions.printSchema()

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 7, Finished, Available, Finished, False)

root
 |-- PatientID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- DateOfBirth: date (nullable = true)
 |-- Gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ContactNumber: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)
 |-- Email_Status: string (nullable = true)
 |-- phone_status: string (nullable = true)
 |-- FirstName_Status: string (nullable = true)
 |-- LastName_Status: string (nullable = true)
 |-- DOB_Status: string (nullable = true)
 |-- Gender_Status: string (nullable = true)
 |-- DataQualityReason: string (nullable = true)
 |-- DataQualityStatus: string (nullable = true)

root
 |-- AdmissionID: integer (nullable = true)
 |-- PatientID: integer (nullable = true)
 |-- AdmissionDate: timestamp (nullable = true)
 |-- DischargeDate: timestamp (nullable = true)
 |-- AdmissionReason: string (n

In [6]:
from pyspark.sql.functions import floor , months_between , current_date

## Creating Age using DOB
gold_patient_summary = (
    patients.select(
        "PatientID",
        "DateOfBirth",
        "Gender",
        "City",
        "State"
    ).withColumn("Age",
    floor(months_between(current_date(),col("DateOfBirth"))/12)
    
    )
)


## Creating AgeGroup using AGE
gold_patient_summary= gold_patient_summary.withColumn(
    "AgeGroup",
    when(col("Age")<18,"Under 18")
    .when(col("Age").between(18,30),"18-30")
    .when(col("Age").between(31,45),"31-45")
    .when(col("Age").between(46,60),"46-60")
    .otherwise("60+")
)



StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import concat_ws

gold_patient_summary = gold_patient_summary.withColumn(
    "Location",
    concat_ws(",",col("city"),col("State"))
)

display(gold_patient_summary)

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6a04c5d0-09bf-40f6-8f1b-25ac695fcd26)

In [8]:
gold_patient_summary=gold_patient_summary.withColumn(
    "AgeGroup",
    when(col("Age")<18,"Child")
    .when(col("Age").between(18,35),"Young Adult")
    .when(col("Age").between(36,50),"Adult")
    .when(col("Age").between(51,65),"Middle Age")
    .otherwise("Senior")
)

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 10, Finished, Available, Finished, False)

In [9]:
gold_patient_summary.groupBy("PatientID") \
    .count() \
    .filter(col("count") > 1) \
    .show()





StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 11, Finished, Available, Finished, False)

+---------+-----+
|PatientID|count|
+---------+-----+
+---------+-----+



In [10]:
gold_patient_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_patient_summary")

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 12, Finished, Available, Finished, False)

#### Gold Admission Summery


In [11]:
display(admissions)

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2624b82a-5cae-454f-b56a-c2368ece23fd)

In [12]:
from pyspark.sql.functions import datediff , date_format ,year , month

gold_admission_summery = (admissions.select(
    "AdmissionID",
        "PatientID",
        "AdmissionDate",
        "DischargeDate",
        "AdmissionReason",
        "Department",
        "AdmissionStatus"

))


gold_admission_summery = gold_admission_summery.withColumn(
    "LengthOfStay",
    datediff(col("DischargeDate"),col("AdmissionDate"))

).withColumn(
    "AdmissionYearMonth" , date_format(col("AdmissionDate"),"yyyy-MM")
).withColumn(
    "AdmissionYear",year(col("AdmissionDate"))
).withColumn(
    "AdmissionMonth",month(col("AdmissionDate"))
)


StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 14, Finished, Available, Finished, False)

In [14]:
gold_admission_summery = gold_admission_summery.withColumn(
    "LengthOfStayCategory",
    when(col("LengthOfStay").isNull(),"Currently Admitted")
    .when(col("LengthOfStay")<=2 ,"Short Stay")
    .when(col("LengthOfStay").between(3,7),"Medium Stay")
    .otherwise("Long Stay")
)

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 16, Finished, Available, Finished, False)

In [15]:
gold_admission_summery.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_admission_summary")

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 17, Finished, Available, Finished, False)

### Gold Mediacaltest

In [16]:
medicaltest.printSchema()

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 18, Finished, Available, Finished, False)

root
 |-- TestID: integer (nullable = true)
 |-- PatientID: integer (nullable = true)
 |-- TestType: string (nullable = true)
 |-- TestResult: string (nullable = true)
 |-- TestDate: timestamp (nullable = true)
 |-- DataQualityReason: string (nullable = true)



In [17]:

gold_medical_test_summary = (
    medicaltest
    .select(
        "TestID",
        "PatientID",
        "TestType",
        "TestResult",
        "TestDate"
    )
)

gold_medical_test_summary = gold_medical_test_summary.withColumn(
    "TestYear",year(col("TestDate"))
).withColumn(
    "TestMonth",month(col("TestDate"))
).withColumn(
    "TestYearMonth",date_format(col("TestDate"),"yyyy-MM")
)



StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 822b7682-b59b-4f1d-b8d9-e4ec2616281c)

In [18]:
gold_medical_test_summary.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_medical_test_summary")

StatementMeta(, de83cbbf-8797-4710-af13-35907972c910, 20, Finished, Available, Finished, False)